# Topic 3.3 - 公共表达式 (Common Table Expression, CTE)

## 1. 公共表达式的概念

大家学 SQL 到此时，应该可以发现一个现象，那就是其实 SQL 的查询结果也是一个表格

- 而且这个表格还很正式，有行有列，列还有列名和数据类型
- 那么我们能不能从这个查询结果这个表格中继续查询呢？
- 答案是可以的，实现这一点有两种方式：**公共表达式**和**子查询**，我们本节来介绍一下公共表达式，下一节我们来介绍一下子查询

公共表达式（Common Table Expression, CTE）是 SQL 中的一种语法结构，它的逻辑其实很简单：

- 那就是在 SQL 中先写一段完整的语句查询出一个临时表
- 然后在这个表的基础上继续写一段 SQL 语句来从这个临时表中查询数据
- 这个临时表就是公共表达式

公共表达式的定义和使用方式如下：

```sql
WITH
临时表1 AS (
    SELECT ...
),
临时表2 AS (
    SELECT ...
),
...
临时表N AS (
    SELECT ...
)
SELECT ...
FROM 临时表1
JOIN 临时表2 ON ...
JOIN ... ON ...
```

- 在这个语法中，`WITH` 关键字后面跟着一个或多个临时表的定义
- 每个临时表的定义都由一个 `SELECT` 语句组成，其中可以有各种子句，例如 `WHERE`、`GROUP BY`、`ORDER BY`、`JOIN`、`UNION` 等等，总之**每个临时表单独拿出来都能执行成一个完整的 SQL 语句抽取出数据**，甚至来说，`WITH` 里还可以嵌套 `WITH`，但可惜 Azure SQL Server 不支持这个功能😅，大家以后进企业如果使用其他类型数据库了，可能就会用到这个功能了
- 临时表可以相互引用，也就是说，在定义临时表的时候，可以使用之前定义的临时表来查询数据，比方说：

    - 在 `临时表2` 的定义中就可以从 `临时表1` 中查询数据
    - 在 `临时表3` 的定义中就可以从 `临时表1` 和 `临时表2` 中查询数据，把他俩 JOIN 在一起或者 UNION 在一起都行
    - 但是有个限制就是只能引用前面定义过的临时表，不能引用后面定义的临时表，否则会报错，例如在 `临时表1` 的定义中引用 `临时表2` 就会报错

- 临时表和临时表的括号之间使用逗号分隔开来，最后一个临时表的括号结束后不加逗号
- 所有临时表定义完成后，最后需要在 `WITH` 外部写一个最终的数据提取语句，这个语句可以使用之前定义的任意一个临时表来查询数据，也可以把多个临时表 JOIN 在一起或者 UNION 在一起都行

这个就是公共表达式的基本定义了，下面我们就来看一些使用公共表达式的例子。

## 2. 构建公共表达式实践

### (1) 定义公共表达式的思路

在 Chinook 数据库中，我们使用公共表达式来做一些练习：我们先来看一个简单的例子：统计每位客户在 Invoice 表中的总消费金额，返回 CustomerId、客户 FirstName 与 LastName、总消费金额，只显示总消费金额大于 40 的客户，并按总消费金额降序排列。

构建 CTE 的一个重要原则就是，一定要确保临时表的查询是正确的
- 因此我们在构建 CTE 的时候，第一步一定要先把每个临时表的查询语句写出来，并且单独执行，确保每个临时表的查询结果都是正确的
- 在这个例子中，我们需要先构建一个临时表 `CustomerSpending`，返回每位客户的总消费金额，我们可以先在 `WITH` 语句中定义这个临时表
- 然后在 `WITH` 外部只需 `SELECT * FROM CustomerSpending` 来查看这个临时表的查询结果，确保这个临时表的查询结果是正确的
- 如果结果太多，可以只查询前几行来查看结果，例如 `SELECT TOP 5 * FROM CustomerSpending`

In [3]:
%%sql
WITH
CustomerSpending AS (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        SUM(I.Total) AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
    GROUP BY C.CustomerId, C.FirstName, C.LastName
)
SELECT Top 5 *
FROM CustomerSpending

,CustomerId,FirstName,LastName,TotalSpending
0,1,Luís,Gonçalves,39.62
1,2,Leonie,Köhler,37.62
2,3,François,Tremblay,39.62
3,4,Bjørn,Hansen,39.62
4,5,František,Wichterlová,40.62


在确保临时表 `CustomerSpending` 的查询结果正确后，我们就可以在这个临时表的基础上继续写 SQL 语句来查询数据了：

In [4]:
%%sql
WITH
CustomerSpending AS (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        SUM(I.Total) AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
    GROUP BY C.CustomerId, C.FirstName, C.LastName
)
SELECT
    CustomerId,
    FirstName,
    LastName,
    TotalSpending
FROM CustomerSpending
WHERE TotalSpending > 40
ORDER BY TotalSpending DESC

,CustomerId,FirstName,LastName,TotalSpending
0,6,Helena,Holý,49.62
1,26,Richard,Cunningham,47.62
2,57,Luis,Rojas,46.62
3,45,Ladislav,Kovács,45.62
4,46,Hugh,O'Reilly,45.62
5,28,Julia,Barnett,43.62
6,37,Fynn,Zimmermann,43.62
7,24,Frank,Ralston,43.62
8,25,Victor,Stevens,42.62
9,7,Astrid,Gruber,42.62


大家在 `.sql` 文件中编写 CTE 代码时，建议把 `WITH` 和 `SELECT *` 这一版保留成草稿不要删除：

- 因为在后续的代码编写中，可能需要大家时不时地回过头来查看一下这个草稿
- 这么做可以帮助确认一下每个临时表的查询结果是否正确，这样可以帮助大家更好地构建 CTE 代码：

```sql
/* CTE的草稿，检查临时表就运行这个 */
WITH
CustomerSpending AS (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        SUM(I.Total) AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
    GROUP BY C.CustomerId, C.FirstName, C.LastName
)
SELECT Top 5 *
FROM CustomerSpending;

/* CTE正式版本 */
WITH
CustomerSpending AS (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        SUM(I.Total) AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
    GROUP BY C.CustomerId, C.FirstName, C.LastName
)
SELECT
    CustomerId,
    FirstName,
    LastName,
    TotalSpending
FROM CustomerSpending
WHERE TotalSpending > 40
ORDER BY TotalSpending DESC;
```

### (2) 通过 CTE 构建复杂 SQL 的实践

接下来我们再来看几个例子：

例1：使用 CTE，先统计每个国家 (BillingCountry) 的平均销售金额，再找出所有金额高于其所在国家平均值的订单，返回：InvoiceId、BillingCountry、Total、国家平均发票金额

- 在这个例子中，我们先构建一个临时表 `CountryAvgInvoice` 来统计每个国家的平均发票金额：

In [5]:
%%sql
WITH
CountryAvgInvoice AS (
    SELECT
        BillingCountry,
        AVG(Total) AS AvgCountryTotal
    FROM Invoice
    GROUP BY BillingCountry
)
SELECT TOP 5 *
FROM CountryAvgInvoice

,BillingCountry,AvgCountryTotal
0,Argentina,5.374285
1,Australia,5.374285
2,Austria,6.088571
3,Belgium,5.374285
4,Brazil,5.431428


- 之后，每个每个订单如何与所在国家的平均发票金额进行比较呢？其实只需把订单表和这个临时表 JOIN 在一起就可以了，以 `BillingCountry` 作为连接键：

In [7]:
%%sql
WITH
CountryAvgInvoice AS (
    SELECT
        BillingCountry,
        AVG(Total) AS AvgCountryTotal
    FROM Invoice
    GROUP BY BillingCountry
)
SELECT TOP 5 *
FROM Invoice           AS I
JOIN CountryAvgInvoice AS C ON C.BillingCountry = I.BillingCountry

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total,BillingCountry.1,AvgCountryTotal
0,119,56,2022-06-12 00:00:00.000,307 Macacha Güemes,Buenos Aires,NaN,Argentina,1106,1.98,Argentina,5.374285
1,142,56,2022-09-14 00:00:00.000,307 Macacha Güemes,Buenos Aires,NaN,Argentina,1106,3.96,Argentina,5.374285
2,164,56,2022-12-17 00:00:00.000,307 Macacha Güemes,Buenos Aires,NaN,Argentina,1106,5.94,Argentina,5.374285
3,216,56,2023-08-07 00:00:00.000,307 Macacha Güemes,Buenos Aires,NaN,Argentina,1106,0.99,Argentina,5.374285
4,337,56,2025-01-28 00:00:00.000,307 Macacha Güemes,Buenos Aires,NaN,Argentina,1106,1.98,Argentina,5.374285


- 这时我们发现，同一行中，就有了订单的发票金额 `Total`，也有了所在国家的平均发票金额 `AvgCountryTotal`，我们就可以直接在 `WHERE` 语句中进行比较了，得到最终结果；

In [8]:
%%sql
WITH
CountryAvgInvoice AS (
    SELECT
        BillingCountry,
        AVG(Total) AS AvgCountryTotal
    FROM Invoice
    GROUP BY BillingCountry
)
SELECT
    I.InvoiceId,
    I.BillingCountry,
    I.Total,
    C.AvgCountryTotal
FROM Invoice           AS I
JOIN CountryAvgInvoice AS C ON C.BillingCountry = I.BillingCountry
WHERE I.Total > C.AvgCountryTotal

,InvoiceId,BillingCountry,Total,AvgCountryTotal
0,164,Argentina,5.94,5.374285
1,348,Argentina,13.86,5.374285
2,403,Argentina,8.91,5.374285
3,305,Australia,8.91,5.374285
4,250,Australia,13.86,5.374285
...,...,...,...,...
167,124,USA,13.86,5.747912
168,115,USA,5.94,5.747912
169,103,USA,15.86,5.747912
170,81,USA,8.91,5.747912


- 通过这个例子，我们可以体会到，不仅仅是 CTE，在 SQL 的构建过程中任何中间步骤都可以先运行出来查看结果，确认结果正确后再继续构建下一步的 SQL 代码，这样可以帮助我们更好地构建 SQL 代码，避免写出错误的 SQL 代码

例2：使用一个或多个 CTE，找出最热销的歌手，返回歌手的名字和该歌手给你带来的总销售金额，并且返回前10名：

- 在这个例子中，我们先构建一个临时表 `ArtistSpending` 来统计每个歌手的总销售金额，这个的逻辑其实就是将好几个表 JOIN 在一起：`InvoiceLine` 表、`Track` 表、`Album` 表、`Artist` 表

In [14]:
%%sql
WITH
ArtistRevenue AS (
    SELECT
        IL.InvoiceLineId,
        A.Name AS ArtistName,
        IL.UnitPrice * IL.Quantity AS Revenue
    FROM Artist      AS A
    JOIN Album       AS B  ON A.ArtistId = B.ArtistId
    JOIN Track       AS T  ON B.AlbumId = T.AlbumId
    JOIN InvoiceLine AS IL ON T.TrackId = IL.TrackId
)
SELECT TOP 5 *
FROM ArtistRevenue

,InvoiceLineId,ArtistName,Revenue
0,1,Accept,0.99
1,2,Accept,0.99
2,3,AC/DC,0.99
3,4,AC/DC,0.99
4,5,AC/DC,0.99


这样，我们就得到了第一个临时表，每个订单项中对应的歌手名字和订单项的销售金额，接下来我们就可以在这个临时表的基础上继续写 SQL 语句来统计每个歌手的总销售金额了：

In [15]:
%%sql
WITH
ArtistRevenue AS (
    SELECT
        IL.InvoiceLineId,
        A.Name AS ArtistName,
        IL.UnitPrice * IL.Quantity AS Revenue
    FROM Artist      AS A
    JOIN Album       AS B  ON A.ArtistId = B.ArtistId
    JOIN Track       AS T  ON B.AlbumId = T.AlbumId
    JOIN InvoiceLine AS IL ON T.TrackId = IL.TrackId
),
ArtistRevenueTotal AS (
    SELECT
        ArtistName,
        SUM(Revenue) AS TotalRevenue
    FROM ArtistRevenue
    GROUP BY ArtistName
)
SELECT TOP 5 *
FROM ArtistRevenueTotal

,ArtistName,TotalRevenue
0,AC/DC,15.84
1,Academy of St. Martin in the Fields & Sir Nevi...,0.99
2,"Academy of St. Martin in the Fields, John Birc...",0.99
3,"Academy of St. Martin in the Fields, Sir Nevil...",1.98
4,Accept,4.95


- 最后，我们在这个基础上继续写 SQL 语句来查询前10名最热销的歌手了：

In [16]:
%%sql
WITH
ArtistRevenue AS (
    SELECT
        IL.InvoiceLineId,
        A.Name AS ArtistName,
        IL.UnitPrice * IL.Quantity AS Revenue
    FROM Artist      AS A
    JOIN Album       AS B  ON A.ArtistId = B.ArtistId
    JOIN Track       AS T  ON B.AlbumId = T.AlbumId
    JOIN InvoiceLine AS IL ON T.TrackId = IL.TrackId
),
ArtistRevenueTotal AS (
    SELECT
        ArtistName,
        SUM(Revenue) AS TotalRevenue
    FROM ArtistRevenue
    GROUP BY ArtistName
)
SELECT TOP 10
    ArtistName,
    TotalRevenue
FROM ArtistRevenueTotal
ORDER BY TotalRevenue DESC

,ArtistName,TotalRevenue
0,Iron Maiden,138.60
1,U2,105.93
2,Metallica,90.09
3,Led Zeppelin,86.13
4,Lost,81.59
5,The Office,49.75
6,Os Paralamas Do Sucesso,44.55
7,Deep Purple,43.56
8,Faith No More,41.58
9,Eric Clapton,39.60


## 3. CTE 综合练习

CTE 属于比较复杂的 SQL 知识了，学好这部分内容需要多练习，下面我们来做一些综合练习，这里我们就不详细展示过程了：

例1：使用 CTE：统计每个客户的发票数量，以及每个客户的总消费金额，最终返回客户姓名、发票数量、总消费金额：

In [19]:
%%sql
WITH
CustomerInvoiceCount AS (
    SELECT
        CustomerId,
        COUNT(*) AS InvoiceCount
    FROM Invoice
    GROUP BY CustomerId
),
CustomerTotalSpending AS (
    SELECT
        CustomerId,
        SUM(Total) AS TotalSpending
    FROM Invoice
    GROUP BY CustomerId
)
SELECT
    C.CustomerId,
    C.FirstName + ' ' + C.LastName AS CustomerName,
    CC.InvoiceCount,
    CS.TotalSpending
FROM      Customer AS C
LEFT JOIN CustomerInvoiceCount  AS CC ON C.CustomerId = CC.CustomerId
LEFT JOIN CustomerTotalSpending AS CS ON C.CustomerId = CS.CustomerId
ORDER BY CS.TotalSpending DESC

,CustomerId,CustomerName,InvoiceCount,TotalSpending
0,6,Helena Holý,7,49.62
1,26,Richard Cunningham,7,47.62
2,57,Luis Rojas,7,46.62
3,45,Ladislav Kovács,7,45.62
4,46,Hugh O'Reilly,7,45.62
5,37,Fynn Zimmermann,7,43.62
6,28,Julia Barnett,7,43.62
7,24,Frank Ralston,7,43.62
8,25,Victor Stevens,7,42.62
9,7,Astrid Gruber,7,42.62


例2：使用 CTE，统计每个国家的客户数量、每个国家的发票数量、以及每个国家的销售总额，最后合并输出：Country、CustomerCount、InvoiceCount
TotalSales，按总销售额降序排列：

In [20]:
%%sql
WITH
CountryCustomers AS (
    SELECT
        Country,
        COUNT(*) AS CustomerCount
    FROM Customer
    GROUP BY Country
),
CountryInvoices AS (
    SELECT
        BillingCountry AS Country,
        COUNT(*) AS InvoiceCount
    FROM Invoice
    GROUP BY BillingCountry
),
CountrySales AS (
    SELECT
        BillingCountry AS Country,
        SUM(Total) AS TotalSales
    FROM Invoice
    GROUP BY BillingCountry
)
SELECT
    CC.Country,
    CC.CustomerCount,
    CI.InvoiceCount,
    CS.TotalSales
FROM CountryCustomers     AS CC
LEFT JOIN CountryInvoices AS CI ON CC.Country = CI.Country
LEFT JOIN CountrySales    AS CS ON CC.Country = CS.Country
ORDER BY CS.TotalSales DESC

,Country,CustomerCount,InvoiceCount,TotalSales
0,USA,13,91,523.06
1,Canada,8,56,303.96
2,France,5,35,195.10
3,Brazil,5,35,190.10
4,Germany,4,28,156.48
5,United Kingdom,3,21,112.86
6,Czech Republic,2,14,90.24
7,Portugal,2,14,77.24
8,India,2,13,75.26
9,Chile,1,7,46.62


例3：使用 CTE，找出销售额最多的的音乐类型（Genre），返回 GenreName、购买次数、总金额：

In [24]:
%%sql
WITH
InvoiceGenre AS (
    SELECT
        I.CustomerId,
        G.Name AS GenreName,
        IL.UnitPrice * IL.Quantity AS Revenue
    FROM InvoiceLine AS IL
    JOIN Invoice     AS I  ON IL.InvoiceId = I.InvoiceId
    JOIN Track       AS T  ON IL.TrackId = T.TrackId
    JOIN Genre       AS G  ON T.GenreId = G.GenreId
),
GenreCount AS (
    SELECT
        GenreName,
        COUNT(*) AS PurchaseCount
    FROM InvoiceGenre
    GROUP BY GenreName
),
GenreRevenue AS (
    SELECT
        GenreName,
        SUM(Revenue) AS TotalRevenue
    FROM InvoiceGenre
    GROUP BY GenreName
)
SELECT
    GC.GenreName,
    GC.PurchaseCount,
    GR.TotalRevenue
FROM GenreCount   AS GC
JOIN GenreRevenue AS GR ON GC.GenreName = GR.GenreName
ORDER BY GR.TotalRevenue DESC

,GenreName,PurchaseCount,TotalRevenue
0,Rock,835,826.65
1,Latin,386,382.14
2,Metal,264,261.36
3,Alternative & Punk,244,241.56
4,TV Shows,47,93.53
5,Jazz,80,79.20
6,Blues,61,60.39
7,Drama,29,57.71
8,R&B/Soul,41,40.59
9,Classical,41,40.59
